# Fridge detector: train on Colab

Runtime: **Runtime > Change runtime type > T4 GPU**.

Colab's own Python is 3.12, but MediaPipe Model Maker needs TensorFlow 2.15 (Python <= 3.11).
So training runs in a separate Python 3.11 environment made with `uv`. The dataset steps
only need PyYAML + Pillow and use Colab's default Python.

Data, runs and exports live on Google Drive so they survive runtime resets.

In [ ]:
# 1. Get the repo (replace with your repo URL; for a private repo use a token or upload a zip)
REPO_URL = "https://github.com/Kronium345/aa-food-vision-model.git"
!git clone -q {REPO_URL} /content/repo
%cd /content/repo

In [ ]:
# 2. Keep data/, runs/ and exports/ on Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/fridge-vision'
for d in ['data', 'runs', 'exports']:
    !mkdir -p {DRIVE}/{d} && rm -rf /content/repo/{d} && ln -s {DRIVE}/{d} /content/repo/{d}
!ls -la /content/repo

In [ ]:
# 3. Python 3.11 training environment (about 3-5 minutes)
!pip -q install uv
!uv venv -q --python 3.11 /content/py311
!uv pip install -q --python /content/py311/bin/python -r requirements-train.lock "tensorflow[and-cuda]==2.15.1"
PY311 = '/content/py311/bin/python'
!{PY311} -c "import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))"

## Data

Skip 4a if `data/raw/openimages` is already on Drive. Put Roboflow exports and your own
CVAT / Label Studio COCO exports under `data/raw/...` and enable them in `datasets.yaml`.

In [ ]:
# 4a. Open Images boxes for the classes that exist there (uses Colab's default Python)
!pip -q install -r requirements-data.txt
!python scripts/download_open_images.py --per-class 800

In [ ]:
# 4b. Merge every enabled source into data/processed/v1 (prints per-class counts + warnings)
!python scripts/export_labels.py
!python scripts/build_dataset.py --version v1

## Train, evaluate, release

In [ ]:
# 4c. Pre-flight before training: repairs an older training venv, migrates older
# datasets (data/ -> images/), checks the imports. Safe to re-run; ends with READY.
!git -C /content/repo pull
!python scripts/colab_preflight.py

In [ ]:
# 5. Train (fp32 + fp16 exports; add --int8 for a quantisation-aware int8 model)
!{PY311} scripts/train.py --data data/processed/v1 --model mobilenet_multi_avg_i384 --epochs 60 --batch-size 16

In [ ]:
# 6. Evaluate on the held-out real-fridge test split
import glob
RUN = sorted(glob.glob('runs/*'))[-1]
print(RUN)
!{PY311} scripts/evaluate.py --model {RUN}/model_fp16.tflite --data data/processed/v1

In [ ]:
# 7. Package the release and download it (copy all three files into the app's assets/models/)
VERSION = 1
!{PY311} scripts/release.py --run {RUN} --variant fp16 --version {VERSION}
from google.colab import files
files.download(f'exports/fridge-detector-v{VERSION}.tflite')
files.download(f'exports/fridge-labels-v{VERSION}.json')
files.download(f'exports/fridge-anchors-v{VERSION}.bin')